# Cyber-Physical Threat Detection: Inference Method Comparison

This notebook implements the full experimental comparison for the paper:
> *Quantum Bayesian Networks for Cyber-Physical Threat Prediction in Pipeline SCADA Systems*

**Bayesian Network:** HIL Water Distribution Testbed (Faramondi et al., *IEEE Access*, 2021)  
**Queries:** 4 diagnostically meaningful posteriors (defined in Phase 1 — Query Registry)  
**Methods compared (by phase):**
- Phase 1: Exact Inference (pgmpy VE), Classical Rejection Sampling (CRS), Quantum Rejection Sampling (QRS)
- Phase 2: Classical Likelihood Weighting (CLW)
- Phase 3: Quantum Likelihood Weighting (QLW)
- Phase 4: Classical Variational Inference (CVI), Quantum Variational Inference (QVI)
- Phase 5: Sensitivity Analysis
- Phase 6: Hardware (IBMQ)

Each stochastic method (CRS, QRS, ...) is run `N_RUNS = 10` times; results reported as mean ± std.

## Setup: Imports, BN Construction, Helpers

In [12]:
import logging
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')
logging.getLogger('qiskit').setLevel(logging.WARNING)
logging.getLogger('qiskit_aer').setLevel(logging.WARNING)

from qubayes.qubayes_tools import Node, Graph, QBNQuery
from qubayes.cyber_attack_bn import (
    ATTACK_ORDER,
    P_A1, A21,
    P_VALVE_S1, P_PUMP_S1, P_TANK_S1,
    P_VALVE_S2, P_FLOW_S2, P_TANK_S2,
)
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.factors.discrete import TabularCPD
from pgmpy.inference import VariableElimination

# ── Experiment parameters ─────────────────────────────────────────────
MASTER_SEED  = 42
N_RUNS       = 10
CRS_SHOTS    = 400
QRS_SHOTS    = 512
QRS_ITERS    = 4      # Grover iterations (fixed; optimal-k analysis in Phase 1c)

rng   = np.random.default_rng(MASTER_SEED)
SEEDS = rng.integers(0, 9999, size=N_RUNS).tolist()
print(f"N_RUNS={N_RUNS}  |  Seeds: {SEEDS}")

# ── State mappings ────────────────────────────────────────────────────
ATTACK_STATES = {"Normal": 0, "MiTM": 1, "DoS": 2, "Physical": 3}
SENSOR_STATES = {"Normal": 0, "Abnormal": 1}

def str_to_idx(node_name, state_str):
    """Map a state string to its integer index for pgmpy."""
    if node_name in ("AttackType_1", "AttackType_2"):
        return ATTACK_STATES[state_str]
    return SENSOR_STATES[state_str]

# ── Emission CPT builder ──────────────────────────────────────────────
def make_emission_cpt_2x4(p_abnormal_by_attack_4, eps=1e-6):
    """
    Build a (2, 4) CPT for a binary sensor node.
    Row 0 = P(Normal|attack),  Row 1 = P(Abnormal|attack).
    Cols: [Normal, MiTM, DoS, Physical].
    """
    p_ab = np.clip(np.array(p_abnormal_by_attack_4, dtype=float), eps, 1.0 - eps)
    cpt  = np.zeros((2, 4), dtype=float)
    cpt[1, :] = p_ab
    cpt[0, :] = 1.0 - p_ab
    assert np.allclose(cpt.sum(axis=0), 1.0), "CPT columns must sum to 1."
    return cpt

# ── QBN construction (for QRS and CRS) ───────────────────────────────
attack1 = Node("AttackType_1", data=P_A1, states=ATTACK_STATES)
attack2 = Node("AttackType_2", parents=["AttackType_1"], data=A21, states=ATTACK_STATES)
valve1  = Node("ValveState_1", parents=["AttackType_1"], data=make_emission_cpt_2x4(P_VALVE_S1), states=SENSOR_STATES)
pump1   = Node("PumpState_1",  parents=["AttackType_1"], data=make_emission_cpt_2x4(P_PUMP_S1),  states=SENSOR_STATES)
tank1   = Node("TankState_1",  parents=["AttackType_1"], data=make_emission_cpt_2x4(P_TANK_S1),  states=SENSOR_STATES)
valve2  = Node("ValveState_2", parents=["AttackType_2"], data=make_emission_cpt_2x4(P_VALVE_S2), states=SENSOR_STATES)
flow2   = Node("FlowState_2",  parents=["AttackType_2"], data=make_emission_cpt_2x4(P_FLOW_S2),  states=SENSOR_STATES)
tank2   = Node("TankState_2",  parents=["AttackType_2"], data=make_emission_cpt_2x4(P_TANK_S2),  states=SENSOR_STATES)

graph = Graph({
    "AttackType_1": attack1, "AttackType_2": attack2,
    "ValveState_1": valve1,  "PumpState_1":  pump1,  "TankState_1": tank1,
    "ValveState_2": valve2,  "FlowState_2":  flow2,  "TankState_2": tank2,
})

qbn_query = QBNQuery()
qbn_query.graph_orig = graph
qbn_query.rebuild_qbn()
print("QBN ready. Nodes:", list(graph.nodes.keys()))

N_RUNS=10  |  Seeds: [892, 7738, 6545, 4388, 4329, 8585, 859, 6972, 2014, 941]
QBN ready. Nodes: ['AttackType_1', 'AttackType_2', 'ValveState_1', 'PumpState_1', 'TankState_1', 'ValveState_2', 'FlowState_2', 'TankState_2']


## Query Registry

Four queries selected to cover three types of causal reasoning over the BN:

| ID | Query | Reasoning Type |
|----|-------|----------------|
| Q1 | P(A2=Physical \| Flow2=Abn, Valve2=Abn) | Forward diagnosis — sensor evidence → S2 attack |
| Q2 | P(A1=Normal \| Flow2=Abn, Valve2=Abn)   | Upstream inference — sensor evidence → S1 cause |
| Q3 | P(A1=Normal \| A2=MiTM)                  | Attack propagation — S2 attack → S1 origin      |
| Q4 | P(A1=MiTM \| A2=MiTM)                    | Attack propagation — S2 attack → S1 origin      |

Q1 and Q2 share the same evidence (rare: P ≈ 0.3%) and test performance under rare-evidence conditions.  
Q3 and Q4 share the same evidence (common: P ≈ 1.2%) and form a complementary pair over A1 states given MiTM in A2.

In [13]:
QUERIES = [
    {
        "id"          : "Q1",
        "label"       : "P(A2=Physical | Flow2=Abn, Valve2=Abn)",
        "evidence"    : {"FlowState_2": "Abnormal", "ValveState_2": "Abnormal"},
        "target"      : {"AttackType_2": "Physical"},
        "target_var"  : "AttackType_2",
        "target_state": "Physical",
    },
    {
        "id"          : "Q2",
        "label"       : "P(A1=Normal | Flow2=Abn, Valve2=Abn)",
        "evidence"    : {"FlowState_2": "Abnormal", "ValveState_2": "Abnormal"},
        "target"      : {"AttackType_1": "Normal"},
        "target_var"  : "AttackType_1",
        "target_state": "Normal",
    },
    {
        "id"          : "Q3",
        "label"       : "P(A1=Normal | A2=MiTM)",
        "evidence"    : {"AttackType_2": "MiTM"},
        "target"      : {"AttackType_1": "Normal"},
        "target_var"  : "AttackType_1",
        "target_state": "Normal",
    },
    {
        "id"          : "Q4",
        "label"       : "P(A1=MiTM | A2=MiTM)",
        "evidence"    : {"AttackType_2": "MiTM"},
        "target"      : {"AttackType_1": "MiTM"},
        "target_var"  : "AttackType_1",
        "target_state": "MiTM",
    },
]
print(f"{len(QUERIES)} queries registered: {[q['id'] for q in QUERIES]}")

4 queries registered: ['Q1', 'Q2', 'Q3', 'Q4']


## Phase 1a — Exact Inference (pgmpy Variable Elimination)

Exact posterior via Variable Elimination serves as ground truth for all comparisons.  
The pgmpy model is constructed from the same CPT constants as the QBN.

In [14]:
# ── Build pgmpy DiscreteBayesianNetwork ───────────────────────────────
pgmpy_model = DiscreteBayesianNetwork([
    ("AttackType_1", "AttackType_2"),
    ("AttackType_1", "ValveState_1"),
    ("AttackType_1", "PumpState_1"),
    ("AttackType_1", "TankState_1"),
    ("AttackType_2", "ValveState_2"),
    ("AttackType_2", "FlowState_2"),
    ("AttackType_2", "TankState_2"),
])

def _sensor_cpd(name, parent, p_abnormal):
    cpt = make_emission_cpt_2x4(p_abnormal)
    return TabularCPD(name, 2, cpt.tolist(),
                      evidence=[parent], evidence_card=[4])

pgmpy_model.add_cpds(
    TabularCPD("AttackType_1", 4, [[v] for v in P_A1.tolist()]),
    TabularCPD("AttackType_2", 4, A21.tolist(),
               evidence=["AttackType_1"], evidence_card=[4]),
    _sensor_cpd("ValveState_1", "AttackType_1", P_VALVE_S1),
    _sensor_cpd("PumpState_1",  "AttackType_1", P_PUMP_S1),
    _sensor_cpd("TankState_1",  "AttackType_1", P_TANK_S1),
    _sensor_cpd("ValveState_2", "AttackType_2", P_VALVE_S2),
    _sensor_cpd("FlowState_2",  "AttackType_2", P_FLOW_S2),
    _sensor_cpd("TankState_2",  "AttackType_2", P_TANK_S2),
)

assert pgmpy_model.check_model(), "pgmpy model validation failed."
ve = VariableElimination(pgmpy_model)
print("pgmpy model ready. CPDs valid.")

pgmpy model ready. CPDs valid.


In [15]:
def exact_inference(q):
    """
    Run Variable Elimination for query q.
    Returns a single float: P(target_var=target_state | evidence).
    """
    evidence_int = {k: str_to_idx(k, v) for k, v in q["evidence"].items()}
    phi = ve.query([q["target_var"]], evidence=evidence_int, show_progress=False)
    idx = str_to_idx(q["target_var"], q["target_state"])
    return float(phi.values[idx])

print("Exact inference results:")
print("-" * 50)
exact_results = {}
for q in QUERIES:
    p = exact_inference(q)
    exact_results[q["id"]] = p
    print(f"  {q['id']}  {q['label']:<45}  {p:.6f}")

Exact inference results:
--------------------------------------------------
  Q1  P(A2=Physical | Flow2=Abn, Valve2=Abn)         0.988239
  Q2  P(A1=Normal | Flow2=Abn, Valve2=Abn)           0.978824
  Q3  P(A1=Normal | A2=MiTM)                         0.633183
  Q4  P(A1=MiTM | A2=MiTM)                           0.362633


## Phase 1b — Multi-Run CRS and QRS

Both methods are run `N_RUNS` times per query with different random seeds.  
Reported metrics:
- **mean ± std** of the posterior estimate across runs
- **acceptance rate** (fraction of shots consistent with evidence) — mean across runs
- **n_valid** — number of runs that returned at least one accepted sample (CRS can return NaN when evidence is very rare)

QRS uses fixed `iterations=4` Grover steps here. Optimal-k analysis follows in Phase 1c.

In [16]:
def run_crs(q, n_runs=N_RUNS, shots=CRS_SHOTS):
    """
    Run classical rejection sampling n_runs times for query q.
    Returns dict of mean, std, acceptance rate, and n_valid.
    """
    qbn_query.evidence = dict(q["evidence"])
    qbn_query.target   = dict(q["target"])
    probs, accs = [], []
    for _ in range(n_runs):
        prob, acc = qbn_query.perform_classical_rejection_sampling(shots=shots)
        probs.append(prob)
        accs.append(acc)
    probs = np.array(probs, dtype=float)
    accs  = np.array(accs,  dtype=float)
    return {
        "mean"   : float(np.nanmean(probs)),
        "std"    : float(np.nanstd(probs)),
        "acc"    : float(np.nanmean(accs)),
        "n_valid": int(np.sum(~np.isnan(probs))),
    }


def run_qrs(q, n_runs=N_RUNS, shots=QRS_SHOTS, iterations=QRS_ITERS, seeds=SEEDS):
    """
    Run quantum rejection sampling n_runs times for query q.
    Rebuilds the QBN circuit before each run (required after prior measurement).
    Returns dict of mean, std, acceptance rate, and n_valid.
    """
    qbn_query.evidence = dict(q["evidence"])
    qbn_query.target   = dict(q["target"])
    probs, accs = [], []
    for seed in seeds[:n_runs]:
        qbn_query.rebuild_qbn()
        prob, acc = qbn_query.perform_rejection_sampling(
            shots=shots, iterations=iterations,
            return_circuit_params=False, seed=int(seed), verbose=False
        )
        probs.append(prob)
        accs.append(acc)
    probs = np.array(probs, dtype=float)
    accs  = np.array(accs,  dtype=float)
    return {
        "mean"   : float(np.nanmean(probs)),
        "std"    : float(np.nanstd(probs)),
        "acc"    : float(np.nanmean(accs)),
        "n_valid": int(np.sum(~np.isnan(probs))),
    }

In [17]:
records = []

for q in QUERIES:
    print(f"\n{'='*60}")
    print(f"{q['id']}  {q['label']}")

    exact = exact_results[q["id"]]
    print(f"  Exact : {exact:.6f}")

    crs = run_crs(q)
    print(f"  CRS   : {crs['mean']:.4f} ± {crs['std']:.4f}  "
          f"acc={100*crs['acc']:.2f}%  valid={crs['n_valid']}/{N_RUNS}")

    qrs = run_qrs(q)
    print(f"  QRS   : {qrs['mean']:.4f} ± {qrs['std']:.4f}  "
          f"acc={100*qrs['acc']:.2f}%  valid={qrs['n_valid']}/{N_RUNS}")

    records.append({
        "id"         : q["id"],
        "label"      : q["label"],
        "exact"      : exact,
        "crs_mean"   : crs["mean"],
        "crs_std"    : crs["std"],
        "crs_acc"    : crs["acc"],
        "crs_n_valid": crs["n_valid"],
        "qrs_mean"   : qrs["mean"],
        "qrs_std"    : qrs["std"],
        "qrs_acc"    : qrs["acc"],
        "qrs_n_valid": qrs["n_valid"],
    })

print(f"\n{'='*60}")
print("All experiments complete.")


Q1  P(A2=Physical | Flow2=Abn, Valve2=Abn)
  Exact : 0.988239
  CRS   : 1.0000 ± 0.0000  acc=1.10%  valid=10/10
  QRS   : 0.9880 ± 0.0046  acc=54.79%  valid=10/10

Q2  P(A1=Normal | Flow2=Abn, Valve2=Abn)
  Exact : 0.978824
  CRS   : 0.9278 ± 0.1030  acc=0.78%  valid=9/10
  QRS   : 0.9788 ± 0.0080  acc=54.79%  valid=10/10

Q3  P(A1=Normal | A2=MiTM)
  Exact : 0.633183
  CRS   : 0.6553 ± 0.0935  acc=2.82%  valid=10/10
  QRS   : 0.6207 ± 0.0206  acc=99.69%  valid=10/10

Q4  P(A1=MiTM | A2=MiTM)
  Exact : 0.362633
  CRS   : 0.3536 ± 0.1426  acc=3.28%  valid=10/10
  QRS   : 0.3752 ± 0.0209  acc=99.69%  valid=10/10

All experiments complete.


## Phase 1 Results Table

The table below is the core result of Phase 1.  
**Error** = |method mean − exact|.  
**Acc%** = mean acceptance rate across runs (higher → more efficient sampling).  
**n_valid** = runs with at least one accepted sample (out of N_RUNS=10; CRS degrades for rare evidence).

In [18]:
df = pd.DataFrame(records)

display_df = pd.DataFrame({
    "Query"          : df["id"],
    "Exact"          : df["exact"].map("{:.4f}".format),
    "CRS (mean±std)" : df.apply(lambda r: f"{r.crs_mean:.4f} ± {r.crs_std:.4f}", axis=1),
    "CRS Err"        : (df["crs_mean"] - df["exact"]).abs().map("{:.4f}".format),
    "CRS Acc%"       : df["crs_acc"].map(lambda x: f"{100*x:.2f}%"),
    "CRS valid"      : df["crs_n_valid"].map(lambda x: f"{x}/{N_RUNS}"),
    "QRS (mean±std)" : df.apply(lambda r: f"{r.qrs_mean:.4f} ± {r.qrs_std:.4f}", axis=1),
    "QRS Err"        : (df["qrs_mean"] - df["exact"]).abs().map("{:.4f}".format),
    "QRS Acc%"       : df["qrs_acc"].map(lambda x: f"{100*x:.2f}%"),
    "QRS valid"      : df["qrs_n_valid"].map(lambda x: f"{x}/{N_RUNS}"),
})

pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', 200)
print(display_df.to_string(index=False))

Query  Exact  CRS (mean±std) CRS Err CRS Acc% CRS valid  QRS (mean±std) QRS Err QRS Acc% QRS valid
   Q1 0.9882 1.0000 ± 0.0000  0.0118    1.10%     10/10 0.9880 ± 0.0046  0.0002   54.79%     10/10
   Q2 0.9788 0.9278 ± 0.1030  0.0510    0.78%      9/10 0.9788 ± 0.0080  0.0000   54.79%     10/10
   Q3 0.6332 0.6553 ± 0.0935  0.0221    2.82%     10/10 0.6207 ± 0.0206  0.0125   99.69%     10/10
   Q4 0.3626 0.3536 ± 0.1426  0.0090    3.28%     10/10 0.3752 ± 0.0209  0.0125   99.69%     10/10


In [ ]:
# ── Acceptance rate speedup ───────────────────────────────────────────
# Key claim: QRS achieves dramatically higher acceptance rates than CRS
# for rare-evidence queries (Q1, Q2).

print("Acceptance rate comparison (QRS vs CRS speedup):")
print("-" * 50)
for row in records:
    crs_acc = row["crs_acc"]
    qrs_acc = row["qrs_acc"]
    speedup = qrs_acc / crs_acc if crs_acc > 0 else float('inf')
    print(f"  {row['id']}  CRS={100*crs_acc:.2f}%  QRS={100*qrs_acc:.2f}%  "
          f"speedup={speedup:.1f}x")

## Phase 1c — Optimal Grover Iterations Analysis

The standard Grover formula gives:
$$k_{\text{opt}} = \left\lfloor \frac{\pi}{4\theta} \right\rfloor, \quad \theta = \arcsin\!\left(\sqrt{P(\text{evidence})}\right)$$

Where $P(\text{evidence})$ is the prior probability of observing the evidence.

**Open question:** Is this formula correct for QBN rejection sampling, where the 'oracle' marks
states consistent with evidence in a joint-distribution-encoded quantum state?
This section:
1. Computes $P(\text{evidence})$ for each query from the joint BN distribution
2. Computes the formula-based $k_{\text{opt}}$
3. Sweeps $k \in \{1, 2, 3, 4, 5, 6, 8, 10\}$ and measures accuracy vs exact for each query
4. Compares empirically optimal $k$ to the formula prediction

This analysis will either validate the formula for QBN or motivate an alternative.
See: Brassard et al. (2002) *Quantum Amplitude Amplification and Estimation* for the theoretical basis.

In [19]:
def compute_p_evidence(q):
    """
    Compute P(evidence) = sum over all configs consistent with evidence
    using pgmpy joint probability.
    We query P(evidence) by marginalizing out all non-evidence variables.
    """
    evidence_int = {k: str_to_idx(k, v) for k, v in q["evidence"].items()}
    # Sum over all states of target variable to get P(evidence)
    phi = ve.query([q["target_var"]], evidence=evidence_int, show_progress=False)
    # phi.values sums to 1 (it's the posterior), so we need P(evidence) separately.
    # Use the joint probability approach: query all evidence nodes jointly.
    # pgmpy VariableElimination doesn't directly return P(evidence),
    # so we use the graph's sample_from_graph + empirical estimate as a check,
    # and compute it analytically via the joint CPT structure.
    #
    # Analytical: P(e) = sum_{all configs} P(all nodes) * 1[config matches evidence]
    joint_phi = ve.query(list(q["evidence"].keys()), evidence={}, show_progress=False)
    # Extract the marginal for the evidence configuration
    idx = tuple(str_to_idx(k, v) for k, v in q["evidence"].items())
    return float(joint_phi.values[idx])


def formula_optimal_k(p_evidence):
    """Standard Grover formula: k_opt = floor(pi / (4 * arcsin(sqrt(p))))."""
    if p_evidence <= 0:
        return float('inf')
    theta = np.arcsin(np.sqrt(p_evidence))
    return max(1, int(np.floor(np.pi / (4 * theta))))


print("P(evidence) and formula-based optimal k per query:")
print("-" * 55)
p_evidence_by_query = {}
for q in QUERIES:
    p_ev = compute_p_evidence(q)
    k_opt = formula_optimal_k(p_ev)
    p_evidence_by_query[q["id"]] = p_ev
    print(f"  {q['id']}  P(evidence)={p_ev:.6f}  theta={np.degrees(np.arcsin(np.sqrt(p_ev))):.2f}°  "
          f"k_opt(formula)={k_opt}")

P(evidence) and formula-based optimal k per query:
-------------------------------------------------------
  Q1  P(evidence)=0.008314  theta=5.23°  k_opt(formula)=8
  Q2  P(evidence)=0.008314  theta=5.23°  k_opt(formula)=8
  Q3  P(evidence)=0.031861  theta=10.28°  k_opt(formula)=4
  Q4  P(evidence)=0.031861  theta=10.28°  k_opt(formula)=4


In [20]:
# Sweep k values and record mean absolute error vs exact for each query
K_VALUES = [1, 2, 3, 4, 5, 6, 8, 10]
K_RUNS   = 5    # fewer runs for the sweep to keep runtime manageable
K_SEEDS  = SEEDS[:K_RUNS]

sweep_records = []
for q in QUERIES:
    exact = exact_results[q["id"]]
    for k in K_VALUES:
        res = run_qrs(q, n_runs=K_RUNS, iterations=k, seeds=K_SEEDS)
        mae = abs(res["mean"] - exact)
        sweep_records.append({
            "query": q["id"], "k": k,
            "mean" : res["mean"], "std": res["std"],
            "acc"  : res["acc"],  "mae": mae,
        })
        print(f"  {q['id']} k={k:2d}: mean={res['mean']:.4f}  mae={mae:.4f}  acc={100*res['acc']:.1f}%")

sweep_df = pd.DataFrame(sweep_records)
print("\nk-sweep complete.")

  Q1 k= 1: mean=0.9905  mae=0.0022  acc=7.7%
  Q1 k= 2: mean=0.9898  mae=0.0016  acc=20.4%
  Q1 k= 3: mean=0.9872  mae=0.0010  acc=36.9%
  Q1 k= 4: mean=0.9880  mae=0.0002  acc=55.2%
  Q1 k= 5: mean=0.9925  mae=0.0043  acc=72.7%
  Q1 k= 6: mean=0.9879  mae=0.0004  acc=87.0%
  Q1 k= 8: mean=0.9898  mae=0.0016  acc=100.0%
  Q1 k=10: mean=0.9860  mae=0.0023  acc=89.5%
  Q2 k= 1: mean=0.9808  mae=0.0020  acc=7.7%
  Q2 k= 2: mean=0.9784  mae=0.0004  acc=20.4%
  Q2 k= 3: mean=0.9765  mae=0.0023  acc=36.9%
  Q2 k= 4: mean=0.9789  mae=0.0001  acc=55.2%
  Q2 k= 5: mean=0.9823  mae=0.0035  acc=72.7%
  Q2 k= 6: mean=0.9798  mae=0.0010  acc=87.0%
  Q2 k= 8: mean=0.9801  mae=0.0012  acc=100.0%
  Q2 k=10: mean=0.9777  mae=0.0011  acc=89.5%
  Q3 k= 1: mean=0.6111  mae=0.0220  acc=27.2%
  Q3 k= 2: mean=0.6211  mae=0.0121  acc=62.8%
  Q3 k= 3: mean=0.6220  mae=0.0112  acc=90.7%
  Q3 k= 4: mean=0.6188  mae=0.0144  acc=99.8%
  Q3 k= 5: mean=0.6233  mae=0.0099  acc=85.5%
  Q3 k= 6: mean=0.6228  mae=0.0103

In [21]:
# Pivot table: MAE at each k value per query
pivot = sweep_df.pivot(index='k', columns='query', values='mae')
pivot.columns.name = None
pivot.index.name   = 'k (Grover iters)'
print("MAE vs exact | lower is better")
print(pivot.round(4).to_string())

print("\nEmpirical best k per query:")
for qid in [q['id'] for q in QUERIES]:
    best_k = sweep_df[sweep_df['query'] == qid].sort_values('mae').iloc[0]
    formula_k = formula_optimal_k(p_evidence_by_query[qid])
    print(f"  {qid}  empirical best k={int(best_k['k'])}  formula k={formula_k}  "
          f"(formula {'matches' if int(best_k['k']) == formula_k else 'does NOT match'})")

MAE vs exact | lower is better
                      Q1      Q2      Q3      Q4
k (Grover iters)                                
1                 0.0022  0.0020  0.0220  0.0235
2                 0.0016  0.0004  0.0121  0.0131
3                 0.0010  0.0023  0.0112  0.0097
4                 0.0002  0.0001  0.0144  0.0143
5                 0.0043  0.0035  0.0099  0.0099
6                 0.0004  0.0010  0.0103  0.0116
8                 0.0016  0.0012  0.1502  0.1460
10                0.0023  0.0011  0.0064  0.0085

Empirical best k per query:
  Q1  empirical best k=4  formula k=8  (formula does NOT match)
  Q2  empirical best k=4  formula k=8  (formula does NOT match)
  Q3  empirical best k=10  formula k=4  (formula does NOT match)
  Q4  empirical best k=10  formula k=4  (formula does NOT match)


## Phase 1 Summary

**Results are in `records` (list of dicts) and `df` (DataFrame).**  
**k-sweep results are in `sweep_df`.**

Phases 2–6 will add columns to `records` as each new method is implemented:
- Phase 2: `clw_mean`, `clw_std`, `clw_acc` (Classical Likelihood Weighting)
- Phase 3: `qlw_mean`, `qlw_std`, `qlw_acc` (Quantum Likelihood Weighting)
- Phase 4: `cvi_mean`, `cvi_std`, `qvi_mean`, `qvi_std` (Variational Inference)
- Phase 5: Sensitivity tables (separate cells)
- Phase 6: Hardware results (separate cells)

## Phase 2 — Classical Likelihood Weighting (CLW)

**Algorithm (Russell & Norvig, Ch. 14):**

1. Process nodes in topological order.
2. For each **non-evidence** node: sample from its CPT given already-sampled parents.
3. For each **evidence** node: fix its value to the observed state; multiply the running weight
   by P(evidence_val | sampled_parents). Do not sample.
4. Every sample contributes — nothing is thrown away.

The posterior estimate is:

$$\hat{P}(\text{target} \mid \text{evidence}) = \frac{\sum_i w_i \cdot \mathbf{1}[\text{sample}_i \text{ matches target}]}{\sum_i w_i}$$

**Why this is better than CRS for rare evidence:**
CRS uses ≈ P(evidence) × N samples effectively.
For Q1/Q2 where P(evidence) ≈ 0.2%, CRS uses ~1 out of 400 shots.
CLW uses all N samples with non-zero weights — no rejection.

**Effective Sample Size (ESS)** measures how many independent samples the weighted set is equivalent to:
$$\text{ESS} = \frac{(\sum_i w_i)^2}{\sum_i w_i^2}$$
ESS/N close to 1 means weights are uniform (ideal). ESS/N ≪ 1 means weight degeneracy
(a few samples dominate), which can happen when evidence is very rare.

**Note on existing code in `qubayes_tools.py`:**
`perform_qbn_likelihood_weighting` and `perform_likelihood_weighted_sampling` both filter samples
to evidence-matching ones first, then weight by P(evidence|parents). This double-counts the
evidence probability and gives no efficiency gain over rejection sampling.
The CLW implementation below is independent of those methods and correct.

In [ ]:
# Pre-compute emission CPT arrays once (shape 2×4, row=sensor state, col=attack type)
CPT = {
    "AttackType_1" : P_A1,                          # shape (4,)
    "AttackType_2" : A21,                            # shape (4, 4)  col = A1 idx
    "ValveState_1" : make_emission_cpt_2x4(P_VALVE_S1),
    "PumpState_1"  : make_emission_cpt_2x4(P_PUMP_S1),
    "TankState_1"  : make_emission_cpt_2x4(P_TANK_S1),
    "ValveState_2" : make_emission_cpt_2x4(P_VALVE_S2),
    "FlowState_2"  : make_emission_cpt_2x4(P_FLOW_S2),
    "TankState_2"  : make_emission_cpt_2x4(P_TANK_S2),
}

# Parent map and topological order for the BN
PARENTS = {
    "AttackType_1": [],
    "AttackType_2": ["AttackType_1"],
    "ValveState_1": ["AttackType_1"],
    "PumpState_1" : ["AttackType_1"],
    "TankState_1" : ["AttackType_1"],
    "ValveState_2": ["AttackType_2"],
    "FlowState_2" : ["AttackType_2"],
    "TankState_2" : ["AttackType_2"],
}
TOPO_ORDER = [
    "AttackType_1", "AttackType_2",
    "ValveState_1", "PumpState_1", "TankState_1",
    "ValveState_2", "FlowState_2", "TankState_2",
]


def _one_clw_sample(evidence_int, rng):
    """
    Draw one weighted sample via classical likelihood weighting.
    Returns (sample dict, weight).
    """
    sample = {}
    w = 1.0
    for node in TOPO_ORDER:
        parents = PARENTS[node]
        cpt = CPT[node]

        # Get conditional probability vector for this node given sampled parents
        if len(parents) == 0:
            probs = cpt                         # shape (n_states,)
        elif len(parents) == 1:
            pa_idx = sample[parents[0]]
            probs = cpt[:, pa_idx]              # shape (n_states,)
        else:
            # Multi-parent (not needed for this BN, included for completeness)
            pa_idx = tuple(sample[p] for p in parents)
            probs = cpt[(...,) + pa_idx]

        probs = np.array(probs, dtype=float)
        probs = probs / probs.sum()             # normalise (guards numerical drift)

        if node in evidence_int:
            # Fix to evidence value — do not sample
            ev_val = evidence_int[node]
            sample[node] = ev_val
            w *= probs[ev_val]                  # accumulate likelihood weight
        else:
            sample[node] = int(rng.choice(len(probs), p=probs))

    return sample, w


def clw_estimate(q, n_samples, seed):
    """
    Classical likelihood weighting for query q.
    Returns (posterior estimate, ESS, ESS/n_samples).
    """
    evidence_int = {k: str_to_idx(k, v) for k, v in q["evidence"].items()}
    target_int   = {k: str_to_idx(k, v) for k, v in q["target"].items()}
    rng = np.random.default_rng(seed)

    weights         = np.zeros(n_samples)
    target_matches  = np.zeros(n_samples, dtype=bool)

    for i in range(n_samples):
        sample, w = _one_clw_sample(evidence_int, rng)
        weights[i]        = w
        target_matches[i] = all(sample[t] == v for t, v in target_int.items())

    w_sum     = weights.sum()
    w_sum_sq  = (weights ** 2).sum()
    ess       = (w_sum ** 2 / w_sum_sq) if w_sum_sq > 0 else 0.0
    posterior = float((weights * target_matches).sum() / w_sum) if w_sum > 0 else np.nan
    return posterior, ess, ess / n_samples


print("CLW implementation ready.")
print("Smoke test (Q1, 1 000 samples):")
p, ess, eff = clw_estimate(QUERIES[0], 1000, seed=42)
print(f"  P(A2=Physical | Flow2=Abn, Valve2=Abn) = {p:.4f}  "
      f"ESS={ess:.1f}  efficiency={100*eff:.1f}%")

In [ ]:
CLW_SAMPLES = 400   # match CRS shot count for a fair comparison

def run_clw(q, n_runs=N_RUNS, n_samples=CLW_SAMPLES, seeds=SEEDS):
    """Run CLW n_runs times; return mean, std, mean ESS, mean efficiency."""
    probs, ess_vals = [], []
    for seed in seeds[:n_runs]:
        p, ess, _ = clw_estimate(q, n_samples, seed=int(seed))
        probs.append(p)
        ess_vals.append(ess)
    probs    = np.array(probs,    dtype=float)
    ess_vals = np.array(ess_vals, dtype=float)
    return {
        "mean"       : float(np.nanmean(probs)),
        "std"        : float(np.nanstd(probs)),
        "ess_mean"   : float(np.nanmean(ess_vals)),
        "ess_eff"    : float(np.nanmean(ess_vals)) / n_samples,
        "n_valid"    : int(np.sum(~np.isnan(probs))),
    }


# Run CLW on all 4 queries and add results to records
print(f"Running CLW ({N_RUNS} runs × {CLW_SAMPLES} samples each)\n")
for rec in records:
    q = next(x for x in QUERIES if x["id"] == rec["id"])
    res = run_clw(q)
    rec["clw_mean"]    = res["mean"]
    rec["clw_std"]     = res["std"]
    rec["clw_ess"]     = res["ess_mean"]
    rec["clw_ess_eff"] = res["ess_eff"]
    rec["clw_n_valid"] = res["n_valid"]
    print(f"  {rec['id']}  {res['mean']:.4f} ± {res['std']:.4f}  "
          f"ESS={res['ess_mean']:.1f} ({100*res['ess_eff']:.1f}%)  "
          f"valid={res['n_valid']}/{N_RUNS}")

In [ ]:
# ── Phase 2 Results: updated comparison table ─────────────────────────
df2 = pd.DataFrame(records)

display_df2 = pd.DataFrame({
    "Query"          : df2["id"],
    "Exact"          : df2["exact"].map("{:.4f}".format),
    "CRS (mean±std)" : df2.apply(lambda r: f"{r.crs_mean:.4f}±{r.crs_std:.4f}", axis=1),
    "CRS valid"      : df2["crs_n_valid"].map(lambda x: f"{x}/{N_RUNS}"),
    "CLW (mean±std)" : df2.apply(lambda r: f"{r.clw_mean:.4f}±{r.clw_std:.4f}", axis=1),
    "CLW ESS%"       : df2["clw_ess_eff"].map(lambda x: f"{100*x:.1f}%"),
    "QRS (mean±std)" : df2.apply(lambda r: f"{r.qrs_mean:.4f}±{r.qrs_std:.4f}", axis=1),
    "QRS Acc%"       : df2["qrs_acc"].map(lambda x: f"{100*x:.1f}%"),
})
pd.set_option('display.max_colwidth', None)
print(display_df2.to_string(index=False))

print("\nCLW vs CRS — std improvement (lower std = more consistent estimates):")
for _, row in df2.iterrows():
    improvement = row.crs_std / row.clw_std if row.clw_std > 0 else float('inf')
    print(f"  {row.id}  CRS std={row.crs_std:.4f}  CLW std={row.clw_std:.4f}  "
          f"{'CLW better' if improvement > 1 else 'CRS better'} ({improvement:.1f}x)")

## Phase 3 — Quantum Likelihood Weighting (QLW)

**Core idea:** Run the QBN circuit *without* Grover amplitude amplification to draw samples from the
joint prior distribution $P(\mathbf{X})$. Then **reweight** every measurement outcome by how
likely the evidence nodes would have taken their observed values, given the sampled parent states.

### Algorithm

1. **Sample from the prior** — run `perform_sampling(shots)` on the QBN circuit.  
   No Grover oracle. Every computational basis state is measured with probability proportional
   to $P(\mathbf{x})$.

2. **Decode each bitstring** — the binarized circuit encodes 4-state nodes across 2 qubits.  
   Sub-node `.0` (MSB) is assigned to the lower qubit index; sub-node `.1` (LSB) to the higher.  
   After reversing Qiskit's bit order, `state_idx = bits[q_low] * 2 + bits[q_high]`.

3. **Compute the likelihood weight** for each outcome:
$$w(\mathbf{x}) = \prod_{e \in \text{evidence}} P(E_e = e_{\text{obs}} \mid \text{parents}(E_e) = \mathbf{x}_{\text{parents}})$$
   No sample is discarded — samples where evidence nodes took a *different* value still contribute,
   but receive a lower weight proportional to $P(e_{\text{other}} \mid \text{parents})$.

4. **Posterior estimate:**
$$\hat{P}(\text{target} \mid \text{evidence}) =
\frac{\displaystyle\sum_{\mathbf{x}} \text{count}(\mathbf{x})\cdot w(\mathbf{x})\cdot\mathbf{1}[\mathbf{x}_{\text{target}} = t]}
     {\displaystyle\sum_{\mathbf{x}} \text{count}(\mathbf{x})\cdot w(\mathbf{x})}$$

### QLW vs CLW

| | CLW | QLW |
|---|---|---|
| Proposal | Classical ancestral sampler | QBN circuit (quantum sampler) |
| Evidence nodes | **Fixed** to observed value | **Free** — sampled by circuit |
| Weighting | $P(e_{\text{obs}} \mid \text{parents})$ | Same formula, applied post-measurement |
| ESS | Low for rare evidence (weight degeneracy) | Similar degeneracy but with quantum speedup potential |

The key difference is the *source* of samples: the QBN circuit exploits quantum superposition to
sample all states simultaneously, making the proposal distribution identical to the classical one
on a noiseless simulator. On real quantum hardware, decoherence introduces a bias that this
comparison will reveal in Phase 6.

**ESS** is computed accounting for repeated bitstrings (grouped shots):
$$\text{ESS} = \frac{\left(\sum_i c_i \cdot w_i\right)^2}{\sum_i c_i \cdot w_i^2}$$
where $c_i$ is the shot count and $w_i$ the likelihood weight for unique bitstring $i$.

In [ ]:
import io
import contextlib

# ─────────────────────────────────────────────────────────────────────────────
# Bitstring decoder
# ─────────────────────────────────────────────────────────────────────────────
# After rebuild_qbn(), the binarized QBN circuit has the following qubit layout
# (for our 8-node BN with no ancillas, 10 qubits total):
#
#   qubit 0  = AttackType_1.0  (MSB of AttackType_1)
#   qubit 1  = AttackType_1.1  (LSB of AttackType_1)
#   qubit 2  = AttackType_2.0  (MSB of AttackType_2)
#   qubit 3  = AttackType_2.1  (LSB of AttackType_2)
#   qubit 4  = ValveState_1
#   qubit 5  = PumpState_1
#   qubit 6  = TankState_1
#   qubit 7  = ValveState_2
#   qubit 8  = FlowState_2
#   qubit 9  = TankState_2
#
# Qiskit returns bitstrings in reverse qubit order (qubit 9 leftmost).
# After bits = list(map(int, bitstring[::-1])), bits[i] = value of qubit i.
#
# For 4-state nodes: state_idx = .0_val * 2 + .1_val
#   i.e. sorted_qubits = [q_for_.0, q_for_.1], bit_str = bits[q_low]+bits[q_high]
#   int(bit_str, 2) = MSB*2 + LSB = state_idx  ✓

def _build_var_qubit_map():
    """
    Build {var_name: sorted_list_of_qubit_indices} from the current qbn_query.qbn.
    Must be called after rebuild_qbn().
    For binary nodes, list has length 1.
    For 4-state nodes, list = [qubit_for_.0 (MSB), qubit_for_.1 (LSB)] sorted ascending.
    """
    var_map = {}
    for var in TOPO_ORDER:
        qubits = sorted(qbn_query.qbn.get_qubits_from_attribute(var))
        var_map[var] = qubits
    return var_map


def _decode_bitstring(bitstring, var_qubit_map):
    """
    Decode a Qiskit measurement bitstring to {var_name: state_int}.

    Qiskit convention: leftmost char = highest qubit index.
    After reversal: bits[i] = value measured at qubit i.

    Binary node  (1 qubit):  state_idx = bits[qubit]
    4-state node (2 qubits): state_idx = bits[q_low]*2 + bits[q_high]
                              = int(str(bits[q_low])+str(bits[q_high]), 2)
    """
    bits = list(map(int, bitstring[::-1]))  # bits[qubit_idx] = measured bit
    sample = {}
    for var, qubits in var_qubit_map.items():
        if len(qubits) == 1:
            sample[var] = bits[qubits[0]]
        else:
            # qubits is already sorted ascending: [q_for_.0(MSB), q_for_.1(LSB)]
            bit_str = ''.join(str(bits[q]) for q in qubits)
            sample[var] = int(bit_str, 2)
    return sample


# ─────────────────────────────────────────────────────────────────────────────
# QLW estimator
# ─────────────────────────────────────────────────────────────────────────────

def qlw_estimate(q, shots, seed):
    """
    Quantum Likelihood Weighting for query q.

    Steps:
      1. Rebuild and run QBN circuit (no Grover) — samples from prior P(X).
      2. Decode each bitstring to classical variable assignments.
      3. Compute per-bitstring weight = Π P(e_obs | sampled_parents(e)).
         All bitstrings contribute; no filtering.
      4. Posterior = Σ(count*w * 1[target_match]) / Σ(count*w).
      5. ESS = (Σ count*w)² / Σ(count*w²)   [grouped-shot formula].

    Returns (posterior, ESS, ESS/shots).
    """
    qbn_query.rebuild_qbn()
    var_qubit_map = _build_var_qubit_map()

    evidence_int = {k: str_to_idx(k, v) for k, v in q["evidence"].items()}
    target_int   = {k: str_to_idx(k, v) for k, v in q["target"].items()}

    # Run circuit — suppress the print() inside perform_sampling
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        result = qbn_query.qbn.perform_sampling(shots=shots, seed=seed)

    total_w      = 0.0
    target_w     = 0.0
    ess_denom    = 0.0   # Σ count*w²  (for grouped-shot ESS)

    for bitstring, count in result.items():
        sample = _decode_bitstring(bitstring, var_qubit_map)

        # Likelihood weight: Π P(e_obs | sampled_parents(e))
        w = 1.0
        for ev_var, ev_val in evidence_int.items():
            parents = PARENTS[ev_var]
            cpt     = CPT[ev_var]
            if len(parents) == 0:
                probs = np.array(cpt, dtype=float)
            elif len(parents) == 1:
                probs = np.array(cpt[:, sample[parents[0]]], dtype=float)
            else:
                pa_idx = tuple(sample[p] for p in parents)
                probs  = np.array(cpt[(...,) + pa_idx], dtype=float)
            probs /= probs.sum()   # guard numerical drift
            w     *= probs[ev_val]

        weighted_count = count * w
        total_w     += weighted_count
        ess_denom   += count * (w ** 2)   # Σ count*w²

        target_match = all(sample[t] == v for t, v in target_int.items())
        if target_match:
            target_w += weighted_count

    if total_w == 0:
        return np.nan, 0.0, 0.0

    posterior = float(target_w / total_w)
    ess       = float((total_w ** 2) / ess_denom) if ess_denom > 0 else 0.0
    return posterior, ess, ess / shots


def run_qlw(q, n_runs=N_RUNS, shots=QRS_SHOTS, seeds=SEEDS):
    """Run QLW n_runs times; return mean, std, mean ESS, mean ESS efficiency."""
    probs, ess_vals = [], []
    for seed in seeds[:n_runs]:
        p, ess, _ = qlw_estimate(q, shots=shots, seed=int(seed))
        probs.append(p)
        ess_vals.append(ess)
    probs    = np.array(probs,    dtype=float)
    ess_vals = np.array(ess_vals, dtype=float)
    return {
        "mean"    : float(np.nanmean(probs)),
        "std"     : float(np.nanstd(probs)),
        "ess_mean": float(np.nanmean(ess_vals)),
        "ess_eff" : float(np.nanmean(ess_vals)) / shots,
        "n_valid" : int(np.sum(~np.isnan(probs))),
    }


# ── Smoke test ────────────────────────────────────────────────────────
print("QLW implementation ready.")
print(f"Shots per run: {QRS_SHOTS}  (same as QRS for fair comparison)\n")
print("Smoke test (Q3 — P(A1=Normal | A2=MiTM), 1 run, seed=42):")
p_smoke, ess_smoke, eff_smoke = qlw_estimate(QUERIES[2], shots=QRS_SHOTS, seed=42)
print(f"  posterior = {p_smoke:.4f}  (exact = {exact_results['Q3']:.4f})")
print(f"  ESS = {ess_smoke:.1f}  efficiency = {100*eff_smoke:.1f}%")

In [ ]:
# ── Run QLW for all 4 queries ─────────────────────────────────────────
print(f"Running QLW ({N_RUNS} runs × {QRS_SHOTS} shots each)\n")
print(f"{'='*65}")

for rec in records:
    q   = next(x for x in QUERIES if x["id"] == rec["id"])
    res = run_qlw(q)
    rec["qlw_mean"]    = res["mean"]
    rec["qlw_std"]     = res["std"]
    rec["qlw_ess"]     = res["ess_mean"]
    rec["qlw_ess_eff"] = res["ess_eff"]
    rec["qlw_n_valid"] = res["n_valid"]

    err = abs(res["mean"] - exact_results[rec["id"]])
    print(f"  {rec['id']}  {res['mean']:.4f} ± {res['std']:.4f}  "
          f"err={err:.4f}  ESS={res['ess_mean']:.1f} ({100*res['ess_eff']:.1f}%)  "
          f"valid={res['n_valid']}/{N_RUNS}")

print(f"\n{'='*65}")
print("QLW complete.")

In [ ]:
# ── Phase 3 Results: full comparison table ───────────────────────────
df3 = pd.DataFrame(records)

display_df3 = pd.DataFrame({
    "Query"          : df3["id"],
    "Exact"          : df3["exact"].map("{:.4f}".format),
    "CRS (mean±std)" : df3.apply(lambda r: f"{r.crs_mean:.4f}±{r.crs_std:.4f}", axis=1),
    "CRS Acc%"       : df3["crs_acc"].map(lambda x: f"{100*x:.1f}%"),
    "CLW (mean±std)" : df3.apply(lambda r: f"{r.clw_mean:.4f}±{r.clw_std:.4f}", axis=1),
    "CLW ESS%"       : df3["clw_ess_eff"].map(lambda x: f"{100*x:.1f}%"),
    "QRS (mean±std)" : df3.apply(lambda r: f"{r.qrs_mean:.4f}±{r.qrs_std:.4f}", axis=1),
    "QRS Acc%"       : df3["qrs_acc"].map(lambda x: f"{100*x:.1f}%"),
    "QLW (mean±std)" : df3.apply(lambda r: f"{r.qlw_mean:.4f}±{r.qlw_std:.4f}", axis=1),
    "QLW ESS%"       : df3["qlw_ess_eff"].map(lambda x: f"{100*x:.1f}%"),
})

pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', 220)
print("Full comparison table (Phases 1–3):")
print(display_df3.to_string(index=False))

# ── ESS comparison: CLW vs QLW ────────────────────────────────────────
print("\nESS comparison (CLW vs QLW) — higher ESS% = less weight degeneracy:")
print("-" * 60)
for _, row in df3.iterrows():
    clw_eff = 100 * row.clw_ess_eff
    qlw_eff = 100 * row.qlw_ess_eff
    better  = "QLW" if qlw_eff > clw_eff else "CLW"
    ratio   = max(qlw_eff, clw_eff) / min(qlw_eff, clw_eff) if min(qlw_eff, clw_eff) > 0 else float('inf')
    print(f"  {row.id}  CLW ESS%={clw_eff:.1f}%  QLW ESS%={qlw_eff:.1f}%  → {better} better ({ratio:.1f}x)")

print("\nError vs exact (|mean - exact|):")
print("-" * 60)
for _, row in df3.iterrows():
    print(f"  {row.id}  "
          f"CRS={abs(row.crs_mean - row.exact):.4f}  "
          f"CLW={abs(row.clw_mean - row.exact):.4f}  "
          f"QRS={abs(row.qrs_mean - row.exact):.4f}  "
          f"QLW={abs(row.qlw_mean - row.exact):.4f}")